# Preprocessing

In [8]:
import pyodbc
import pandas as pd

In [9]:
df_test = pd.read_csv('../../../decoded_data/BA51/FactTest.csv')
df_question = pd.read_csv('../../../decoded_data/BA51/FactQuestionBAQ.csv')

In [10]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71469 entries, 0 to 71468
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Test             71469 non-null  object 
 1   CandidateKey     71469 non-null  int64  
 2   CreatedDateKey   62120 non-null  float64
 3   ModifiedDateKey  62126 non-null  float64
 4   VersionNumber    69221 non-null  object 
 5   TestKey          71469 non-null  int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 3.3+ MB


In [11]:
df_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 469187 entries, 0 to 469186
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   NormItemID   469187 non-null  int64 
 1   Norm1        469187 non-null  int64 
 2   Norm2        469187 non-null  int64 
 3   Norm3        469187 non-null  int64 
 4   Norm4        469187 non-null  int64 
 5   Norm5        469187 non-null  int64 
 6   IpsItemID    469187 non-null  int64 
 7   Ips1         469187 non-null  int64 
 8   Ips2         469187 non-null  int64 
 9   Ips3         469187 non-null  int64 
 10  Ips4         469187 non-null  int64 
 11  Ips5         469187 non-null  int64 
 12  Test         469187 non-null  object
 13  QuestionKey  469187 non-null  int64 
 14  TestKey      469187 non-null  int64 
dtypes: int64(14), object(1)
memory usage: 53.7+ MB


In [12]:
df_test.drop(columns=['VersionNumber', 'Test', 'ModifiedDateKey', 'CreatedDateKey'], inplace=True)
df_test.head()

,CandidateKey,TestKey
0,1034431,172548
1,1034091,172549
2,1033721,172550
3,1025991,172551
4,1034191,172552


In [13]:
df_question.drop(columns=['Test'], inplace=True)
df_question.head()

,NormItemID,Norm1,Norm2,Norm3,Norm4,Norm5,IpsItemID,Ips1,Ips2,Ips3,Ips4,Ips5,QuestionKey,TestKey
0,1,2,3,5,1,4,1,2,3,4,1,5,1,172548
1,1,4,5,1,2,3,1,4,5,1,2,3,2,172549
2,1,4,5,2,1,3,1,5,4,2,1,3,3,172550
3,1,4,3,1,2,5,1,4,3,1,2,5,4,172551
4,1,1,2,5,4,3,1,2,1,5,4,3,5,172551


In [14]:
df = df_question.merge(df_test, on='TestKey')
df.head()

,NormItemID,Norm1,Norm2,Norm3,Norm4,Norm5,IpsItemID,Ips1,Ips2,Ips3,Ips4,Ips5,QuestionKey,TestKey,CandidateKey
0,1,2,3,5,1,4,1,2,3,4,1,5,1,172548,1034431
1,1,4,5,1,2,3,1,4,5,1,2,3,2,172549,1034091
2,1,4,5,2,1,3,1,5,4,2,1,3,3,172550,1033721
3,1,4,3,1,2,5,1,4,3,1,2,5,4,172551,1025991
4,1,1,2,5,4,3,1,2,1,5,4,3,5,172551,1025991


## Apart ipsative en normative

In [15]:
# DataFrame met alleen de Norm-kolommen
df_norm = df[['TestKey','NormItemID', 'Norm1', 'Norm2', 'Norm3', 'Norm4', 'Norm5']]

# DataFrame met alleen de Ips-kolommen
df_ips = df[['TestKey', 'IpsItemID', 'Ips1', 'Ips2', 'Ips3', 'Ips4', 'Ips5']]

In [16]:
df_norm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 469187 entries, 0 to 469186
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   TestKey     469187 non-null  int64
 1   NormItemID  469187 non-null  int64
 2   Norm1       469187 non-null  int64
 3   Norm2       469187 non-null  int64
 4   Norm3       469187 non-null  int64
 5   Norm4       469187 non-null  int64
 6   Norm5       469187 non-null  int64
dtypes: int64(7)
memory usage: 25.1 MB


In [17]:
df_ips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 469187 entries, 0 to 469186
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   TestKey    469187 non-null  int64
 1   IpsItemID  469187 non-null  int64
 2   Ips1       469187 non-null  int64
 3   Ips2       469187 non-null  int64
 4   Ips3       469187 non-null  int64
 5   Ips4       469187 non-null  int64
 6   Ips5       469187 non-null  int64
dtypes: int64(7)
memory usage: 25.1 MB


## Cleaning

### Normative

In [18]:
df_melted_norm = df_norm.melt(id_vars=['TestKey', 'NormItemID'], 
                    value_vars=['Norm1', 
                                'Norm2',
                                'Norm3',
                                'Norm4',
                                'Norm5'], 
                    var_name='answer_type', 
                    value_name='Answer')
df_melted_norm['QuestionNr'] = df_melted_norm['answer_type'].str.extract(r'(\d+)', expand=False).astype('Int8')
df_melted_norm.drop(columns=["answer_type"], inplace=True)

In [19]:
df_melted_norm

,TestKey,NormItemID,Answer,QuestionNr
0,172548,1,2,1
1,172549,1,4,1
2,172550,1,4,1
3,172551,1,4,1
4,172551,1,1,1
...,...,...,...,...
2345930,244016,62,5,5
2345931,244016,3,5,5
2345932,244016,8,5,5
2345933,244016,13,5,5


In [20]:
df_pivot_norm = df_melted_norm.pivot_table(index='TestKey', 
                                columns=['NormItemID', 'QuestionNr'], 
                                values='Answer', 
                                aggfunc='first')
df_pivot_norm.columns = [f'Q{q}_A{a}_norm' for q, a in df_pivot_norm.columns]
df_pivot_norm.fillna(0.0, inplace=True)

In [21]:
df_pivot_norm

,Q0_A1_norm,Q0_A2_norm,Q0_A3_norm,Q0_A4_norm,Q0_A5_norm,Q1_A1_norm,Q1_A2_norm,Q1_A3_norm,Q1_A4_norm,Q1_A5_norm,...,Q62_A1_norm,Q62_A2_norm,Q62_A3_norm,Q62_A4_norm,Q62_A5_norm,Q63_A1_norm,Q63_A2_norm,Q63_A3_norm,Q63_A4_norm,Q63_A5_norm
TestKey,,,,,,,,,,,,,,,,,,,,,
172548,0.0,0.0,0.0,0.0,0.0,2.0,3.0,5.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172549,0.0,0.0,0.0,0.0,0.0,4.0,5.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172550,0.0,0.0,0.0,0.0,0.0,4.0,5.0,2.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172551,0.0,0.0,0.0,0.0,0.0,4.0,3.0,1.0,2.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172552,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244012,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,5.0,...,1.0,2.0,3.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0
244013,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,5.0,...,1.0,2.0,3.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0
244014,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,5.0,...,1.0,2.0,3.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0


### Ipsatives

In [22]:
df_melted_ips = df_ips.melt(id_vars=['TestKey', 'IpsItemID'], 
                    value_vars=['Ips1', 
                                'Ips2',
                                'Ips3',
                                'Ips4',
                                'Ips5'], 
                    var_name='answer_type', 
                    value_name='Answer')
df_melted_ips['QuestionNr'] = df_melted_ips['answer_type'].str.extract(r'(\d+)', expand=False).astype('Int8')
df_melted_ips.drop(columns=["answer_type"], inplace=True)

In [23]:
df_pivot_ips = df_melted_ips.pivot_table(index='TestKey', 
                                columns=['IpsItemID', 'QuestionNr'], 
                                values='Answer', 
                                aggfunc='first')
df_pivot_ips.columns = [f'Q{q}_A{a}_ips' for q, a in df_pivot_ips.columns]
df_pivot_ips.fillna(0.0, inplace=True)

In [24]:
df_pivot_ips

,Q0_A1_ips,Q0_A2_ips,Q0_A3_ips,Q0_A4_ips,Q0_A5_ips,Q1_A1_ips,Q1_A2_ips,Q1_A3_ips,Q1_A4_ips,Q1_A5_ips,...,Q62_A1_ips,Q62_A2_ips,Q62_A3_ips,Q62_A4_ips,Q62_A5_ips,Q63_A1_ips,Q63_A2_ips,Q63_A3_ips,Q63_A4_ips,Q63_A5_ips
TestKey,,,,,,,,,,,,,,,,,,,,,
172548,0.0,0.0,0.0,0.0,0.0,2.0,3.0,4.0,1.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172549,0.0,0.0,0.0,0.0,0.0,4.0,5.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172550,0.0,0.0,0.0,0.0,0.0,5.0,4.0,2.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172551,0.0,0.0,0.0,0.0,0.0,4.0,3.0,1.0,2.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172552,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244012,0.0,0.0,0.0,0.0,0.0,1.0,3.0,3.0,3.0,5.0,...,1.0,3.0,3.0,3.0,5.0,0.0,0.0,0.0,0.0,0.0
244013,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,5.0,...,1.0,2.0,3.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0
244014,0.0,0.0,0.0,0.0,0.0,3.0,3.0,5.0,1.0,3.0,...,1.0,3.0,3.0,3.0,5.0,0.0,0.0,0.0,0.0,0.0


In [25]:
df_pivot_ips.drop_duplicates(inplace=True)
df_pivot_norm.drop_duplicates(inplace=True)

In [26]:
df_pivot_norm

,Q0_A1_norm,Q0_A2_norm,Q0_A3_norm,Q0_A4_norm,Q0_A5_norm,Q1_A1_norm,Q1_A2_norm,Q1_A3_norm,Q1_A4_norm,Q1_A5_norm,...,Q62_A1_norm,Q62_A2_norm,Q62_A3_norm,Q62_A4_norm,Q62_A5_norm,Q63_A1_norm,Q63_A2_norm,Q63_A3_norm,Q63_A4_norm,Q63_A5_norm
TestKey,,,,,,,,,,,,,,,,,,,,,
172548,0.0,0.0,0.0,0.0,0.0,2.0,3.0,5.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172549,0.0,0.0,0.0,0.0,0.0,4.0,5.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172550,0.0,0.0,0.0,0.0,0.0,4.0,5.0,2.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172551,0.0,0.0,0.0,0.0,0.0,4.0,3.0,1.0,2.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172552,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244004,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,5.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244007,0.0,0.0,0.0,0.0,0.0,3.0,5.0,4.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
df_pivot_ips

,Q0_A1_ips,Q0_A2_ips,Q0_A3_ips,Q0_A4_ips,Q0_A5_ips,Q1_A1_ips,Q1_A2_ips,Q1_A3_ips,Q1_A4_ips,Q1_A5_ips,...,Q62_A1_ips,Q62_A2_ips,Q62_A3_ips,Q62_A4_ips,Q62_A5_ips,Q63_A1_ips,Q63_A2_ips,Q63_A3_ips,Q63_A4_ips,Q63_A5_ips
TestKey,,,,,,,,,,,,,,,,,,,,,
172548,0.0,0.0,0.0,0.0,0.0,2.0,3.0,4.0,1.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172549,0.0,0.0,0.0,0.0,0.0,4.0,5.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172550,0.0,0.0,0.0,0.0,0.0,5.0,4.0,2.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172551,0.0,0.0,0.0,0.0,0.0,4.0,3.0,1.0,2.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172552,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244004,0.0,0.0,0.0,0.0,0.0,1.0,2.0,4.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244007,0.0,0.0,0.0,0.0,0.0,4.0,5.0,3.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
df_alles = df_pivot_norm.merge(df_pivot_ips, on=['TestKey'])

In [29]:
df_alles

,Q0_A1_norm,Q0_A2_norm,Q0_A3_norm,Q0_A4_norm,Q0_A5_norm,Q1_A1_norm,Q1_A2_norm,Q1_A3_norm,Q1_A4_norm,Q1_A5_norm,...,Q62_A1_ips,Q62_A2_ips,Q62_A3_ips,Q62_A4_ips,Q62_A5_ips,Q63_A1_ips,Q63_A2_ips,Q63_A3_ips,Q63_A4_ips,Q63_A5_ips
TestKey,,,,,,,,,,,,,,,,,,,,,
172548,0.0,0.0,0.0,0.0,0.0,2.0,3.0,5.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172549,0.0,0.0,0.0,0.0,0.0,4.0,5.0,1.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172550,0.0,0.0,0.0,0.0,0.0,4.0,5.0,2.0,1.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172551,0.0,0.0,0.0,0.0,0.0,4.0,3.0,1.0,2.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
172552,0.0,0.0,0.0,0.0,0.0,1.0,5.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244004,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,5.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
244007,0.0,0.0,0.0,0.0,0.0,3.0,5.0,4.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
# df_dupl.set_index('TestKey', inplace=True)
df_pivot_ips.to_csv('../csv/preprocessed_data_ips.csv')
df_pivot_norm.to_csv('../csv/preprocessed_data_norm.csv')
df_alles.to_csv('../csv/preprocessed_data.csv')